In [ ]:
!pip install nltk ollama rouge-score

In [10]:
from ollama import Client
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

ollama = Client(host = 'http://localhost:11434')

In [11]:
context = """
2025-10-15 12:02:31 ERROR ConnectionTimeout: Database connection failed after 30s.
2025-10-15 12:02:32 INFO Retrying connection...
2025-10-15 12:02:35 ERROR AuthenticationFailed: Invalid DB credentials.
2025-10-15 12:02:40 INFO Shutting down pipeline gracefully.
"""

reference_summary = "Database connection failed due to timeout and authentication issues."

In [65]:
def ask_model(prompt):
    res = ollama.chat(model = 'mistral',messages = [{'role':'user','content':prompt}])
    return res['message']['content'].strip()

In [13]:
#Zero-Shot, Give straight answers, we dont give any example to it
zero_shot_prompt = f"Summarize the following server log in one sentence:\n{context}"
zero_shot_output = ask_model(zero_shot_prompt)
zero_shot_output

'The server log indicates that there was a connection timeout to the database, followed by an authentication failure due to invalid credentials, and the system is now shutting down the pipeline gracefully.'

In [ ]:
#One-Shot ,Better Understanding, We give one example to it and gives the answers as per our example
one_shot_prompt = f"""
Example:
Log: "2025-10-14 08:01:10 ERROR APIError: Token expired."
Summary: API request failed due to expired token.

Now summarize the following:
{context}
"""

one_shot_output = ask_model(one_shot_prompt)
one_shot_output

#Few-shot
few_shot_prompt = f"""
Examples:
Log: "2025-10-14 08:01:10 ERROR APIError: Token expired."
Summary: API request failed due to expired token.
---
Log: "2025-10-12 22:10:05 ERROR DiskFull: Cannot write to /tmp."
Summary: Disk was full preventing file writes.
---
Now summarize the following:
{context}
"""

few_shot_output = ask_model(few_shot_prompt)
few_shot_output

#Chain of Thought
cot_prompt = f"""
Let's reason step by step.
1. Identify key errors and their causes.
2. Summarize them concisely.
Logs:
{context}
"""
cot_output = ask_model(cot_prompt)
print(cot_output)

#Self-Consistency : Multiple reasoning samples averaged
import random

sc_outputs = []
for i in range(3):  # 3 Reasoning Paths
    sc_prompt = cot_prompt + f'\nReasoning attempt {i+1}:'
    sc_outputs.append(ask_model(sc_prompt))

#choose most frequent / best overlap summary (simple hueristic)
from collections import Counter
final_sc_output = Counter(sc_outputs).most_common(1)[0][0]
print(final_sc_output)

Counter(['hi','hello','hi']).most_common()

print(sc_outputs)

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
def evaluate(prediction,reference):
    reference_tokens = [reference.split()]
    prediction_token =[prediction.split()]
    try:
        bleu = corpus_bleu([reference_tokens],[prediction_token], smoothing_function=SmoothingFunction().method1)
    except TypeError:
        bleu = 0.0
    
    #ROUGE
        rouge = rouge_scorer.RougeScorer(['rouge1','rougeL'],use_stemmer = True)
        rouge_scores = rouge.score(reference,prediction)
        rouge1 = rouge_scores['rouge1'].fmeasure
        rougeL = rouge_scores['rougeL'].fmeasure
    
        return bleu, rouge1, rougeL


for label, output in [
    ("Zero-shot", zero_shot_output),
    ("One-shot", one_shot_output),
    ("Few-shot", few_shot_output),
    ("Chain-of-Thought", cot_output),
    ("Self-Consistency", final_sc_output),
]:
    bleu,rouge1,rougeL = evaluate(output,reference_summary)
    print(f"{label}: \nOutput: {output}\nBLEU={bleu:.3f}, ROUGE-1 = {rouge1:.3f},ROUGE-L = {rougeL:.3f}\n")
